# Spotify Machine Learning

Recreates two workflows from [Spotify-Machine-Learning](https://github.com/alejandrolucchesi/Spotify-Machine-Learning):

1. **K-means clustering** → split songs into two playlist-style groups
2. **Mood classification** → predict Calm / Energetic / Happy / Sad from audio features

In [ ]:
from pathlib import Path
import sys

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from src.clustering import load_clustering_data, run_clustering, cluster_summary, save_cluster_outputs
from src.mood_model import load_mood_data, train_mood_classifier, predict_mood_from_features, save_mood_model

## 1. Clustering (K-means, k=2)

In [ ]:
df = load_clustering_data("data/df1.csv", "data/df2.csv")
clustered, kmeans, scaler = run_clustering(df)
save_cluster_outputs(clustered, "data")
cluster_summary(clustered)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["danceability", "energy", "valence", "loudness"]):
    ax.hist(df[col], bins=30)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    clustered["energy"],
    clustered["danceability"],
    clustered["loudness"],
    c=clustered["kmeans"],
    cmap="bwr",
    s=40,
)
ax.set_xlabel("Energy")
ax.set_ylabel("Danceability")
ax.set_zlabel("Loudness")
ax.set_title("Songs clustered (k=2)")
plt.show()

## 2. Mood classification

In [ ]:
moods = load_mood_data("data/data_moods.csv")
pipeline, encoder, metrics = train_mood_classifier(moods)
save_mood_model(pipeline, encoder, "artifacts/mood_model.joblib")

print(f"CV accuracy: {metrics['cv_mean_accuracy']:.2%}")
print(f"Test accuracy: {metrics['test_accuracy']:.2%}")
print(metrics["classification_report"])

In [ ]:
import numpy as np

cm = np.array(metrics["confusion_matrix"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Mood confusion matrix")
plt.show()

In [ ]:
sample = moods.iloc[0]
pred = predict_mood_from_features(sample, pipeline, encoder)
print(f"{sample['name']} by {sample['artist']}: predicted {pred}, actual {sample['mood']}")